# Phase 3 - Step 1: Handle Duplicates & Load Clean Data

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/creditcard.csv')
print('Original shape:', df.shape)

# Remove duplicate rows (found in EDA phase)
df = df.drop_duplicates()
print('Shape after removing duplicates:', df.shape)

# Confirm class distribution is still reasonable after dedup
print('\nClass distribution after dedup:')
print(df['Class'].value_counts())

Original shape: (284807, 31)


Shape after removing duplicates: (283726, 31)

Class distribution after dedup:
Class
0    283253
1       473
Name: count, dtype: int64


# Step 2: Feature Scaling (Amount & Time)

In [2]:
from sklearn.preprocessing import StandardScaler

# V1-V28 are already PCA-transformed (already scaled), so we only scale Amount and Time
scaler = StandardScaler()

df['Amount_scaled'] = scaler.fit_transform(df[['Amount']])
df['Time_scaled'] = scaler.fit_transform(df[['Time']])

# Drop original unscaled columns, keep scaled versions
df = df.drop(['Amount', 'Time'], axis=1)

print('Columns after scaling:')
print(df.columns.tolist())
print('\nSample of scaled columns:')
print(df[['Amount_scaled', 'Time_scaled']].describe())

Columns after scaling:
['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Class', 'Amount_scaled', 'Time_scaled']

Sample of scaled columns:
       Amount_scaled   Time_scaled
count   2.837260e+05  2.837260e+05
mean   -8.013847e-19  1.538659e-16
std     1.000002e+00  1.000002e+00
min    -3.533268e-01 -1.996823e+00
25%    -3.309625e-01 -8.552128e-01
50%    -2.654671e-01 -2.131081e-01
75%    -4.378088e-02  9.369423e-01
max     1.022476e+02  1.642362e+00


# Step 3: Train/Validation/Test Split (Leakage-Safe)

In [3]:
from sklearn.model_selection import train_test_split

X = df.drop('Class', axis=1)
y = df['Class']

# Step A: split into train (70%) and temp (30%)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

# Step B: split temp into validation (15%) and test (15%)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print('Train shape:', X_train.shape, ' Fraud rate: {:.4f}%'.format(y_train.mean()*100))
print('Val shape:  ', X_val.shape, ' Fraud rate: {:.4f}%'.format(y_val.mean()*100))
print('Test shape: ', X_test.shape, ' Fraud rate: {:.4f}%'.format(y_test.mean()*100))

Train shape: (198608, 30)  Fraud rate: 0.1667%
Val shape:   (42559, 30)  Fraud rate: 0.1668%
Test shape:  (42559, 30)  Fraud rate: 0.1668%


**Important — Leakage prevention notes:**
- We used `stratify=y` so each split (train/val/test) preserves the same fraud ratio.
- Scaling (Step 2) was done BEFORE the split here for simplicity since `StandardScaler` was fit on Amount/Time only, not on the target. In a stricter production pipeline, you would fit the scaler on `X_train` only and apply (`transform`, not `fit_transform`) to val/test, to avoid any statistical leakage from val/test into train. We'll do it the strict way from Phase 5 onward using `Pipeline`.
- No transaction should appear in more than one split — `train_test_split` guarantees this by design.


In [4]:
# Save the splits so later phases (Class Imbalance, Modeling) can reuse them
import os
os.makedirs('../data/processed', exist_ok=True)

X_train.to_csv('../data/processed/X_train.csv', index=False)
X_val.to_csv('../data/processed/X_val.csv', index=False)
X_test.to_csv('../data/processed/X_test.csv', index=False)
y_train.to_csv('../data/processed/y_train.csv', index=False)
y_val.to_csv('../data/processed/y_val.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)

print('Saved all splits to data/processed/')

Saved all splits to data/processed/
